# Lexos API Plotly Clustermap Tutorial

### 1. Setting Up Your Environment: The Essential Tools
 Before we dive into analyzing texts, we need to make sure you have the necessary "tools" installed on your computer. These are software libraries that `lexos` relies on to do its work, particularly `plotly` for interactive visualizations.
 
 **Important:** These commands need to be run in your command line (also known as your terminal or console), not directly in this Jupyter Notebook. Open your command line program and type these commands, pressing Enter after each one:


### 2. Loading Our Libraries: Getting Ready to Work
 Now that the tools are installed, let's load them into our Jupyter Notebook environment.
 Run the following code cell:


In [ ]:
import spacy
import pandas as pd
import numpy as np
from pathlib import Path

# Assuming lexos is installed, or the lexos folder is in your Python path
# For this tutorial, we'll assume the `lexos` package is available.
try:
    from lexos.dtm import DTM
    from lexos.cluster.plotly_clustermap import PlotlyClustermap
    from lexos.exceptions import LexosException
except ImportError:
    print("Could not import Lexos modules. Make sure 'lexos' is installed or accessible.")
    print("You might need to run: pip install lexos")

nlp = spacy.load("en_core_web_sm")

### 3. Gathering Our Texts: The Documents for Analysis
 
 Next, we need to tell `lexos` which text files we want to analyze. For this tutorial, we'll use a few classic literary works as examples. We'll store their locations in a Python dictionary.
 
 A **dictionary** is like a list where each item has a unique **label** (like "Poe" or "Irving") that helps us find its corresponding **value** (the actual path to the text file on your computer). These labels will become the names of your documents.
 
 **Important Note for Your Own Texts:** If you replace these example files with your own custom texts, please be aware that `spaCy` (the text processing library) has a default safety limit of **1,000,000 characters** per document.
 
 **Before running this cell, make sure the file paths are correct for where you've saved these example texts on your computer!** If you move this Jupyter Notebook, you might need to adjust the paths to point to the correct locations.


In [ ]:
from pathlib import Path

# Load text files from uploaded documents.
# The 'key' (e.g., "Poe") will be our document label.
# The 'value' is the path to the text file.
files = {
    "Poe": "FilesToUse/Poe_FallOfHouseUsher_1839.txt",
    "Lippard": "FilesToUse/Lippard_BelOfPrairieEden.txt",
    "Irving": "FilesToUse/Irving_RipVanWInkle.txt",
    "Henry": "FilesToUse/HenryWP_ThePirate.txt",
}

# Read the content of each text file into a list of strings
docs = [Path(path).read_text(encoding="utf-8") for path in files.values()]

# Get the labels (names) for each document directly from our dictionary's keys
labels = list(files.keys())

print(f"Loaded {len(docs)} documents with labels: {labels}")

Loaded 4 documents with labels: ['Poe', 'Lippard', 'Irving', 'Henry']


### 4. Creating the Document-Term Matrix (DTM): Our Linguistic "Spreadsheet"
 
 Before we can build a clustermap, we need to transform our raw text documents into a structured numerical format that computers can understand. This is where the **Document-Term Matrix (DTM)** comes in.
 
 Imagine a giant spreadsheet (a matrix) where:
 * Each **row** represents one of your documents (e.g., "Poe", "Irving").
 * Each **column** represents a unique "term" (usually a word) found across all your documents.
 * Each **cell** contains a number indicating how many times that specific "term" appears in that specific document.
 
 This matrix is the foundation for analyzing textual relationships.

 #### What is a "Term" and How Do We Prepare Our Texts?
 
 In this context, a "term" is typically a word. However, before counting words, `lexos` (using `spaCy` behind the scenes) performs several crucial **pre-processing** steps. These steps clean and standardize your text, ensuring that the counts are meaningful for analysis.
 
 Here are some important pre-processing options you can control when creating the DTM:
 
 * **`stop_words`**: These are very common words (like 'a', 'an', 'the', 'is', 'and', 'but') that appear frequently in almost all texts and often don't carry much unique meaning for distinguishing between documents.
     * **`True` (default)**: Removes these common words.
     * **`False`**: Keeps all words.
     * *Why use it?* Removing stop words helps you focus on the more distinctive vocabulary that truly characterizes your texts.
 
 * **`lemmatize`**: This reduces words to their **base form** (their "lemma"). For example, 'running', 'ran', and 'runs' would all become 'run'.
     * **`True`**: Words are lemmatized.
     * **`False` (default)**: Words are not lemmatized.
     * *Why use it?* If you want to analyze concepts regardless of their grammatical variations, lemmatization is very useful. For instance, focusing on the concept of "fear" rather than distinguishing between "fear," "fears," and "feared."
 
 * **`pos_filter`**: This allows you to include only terms that are specific **Parts-of-Speech (POS)**, like nouns, verbs, adjectives, etc.
     * You provide a list of desired POS tags (e.g., `['NOUN', 'VERB', 'ADJ']`).
     * *Why use it?* If you're interested in how documents relate based only on the **objects** they discuss (nouns), the **actions** they describe (verbs), or the **qualities** they assign (adjectives), this filter is incredibly powerful.
 
 * **`min_freq` / `max_freq`**: These settings control the frequency of terms included in the DTM.
     * **`min_freq`**: The minimum number of times a term must appear *across all documents* to be included. Setting `min_freq=2` would remove unique misspellings or rare words that might not be significant.
     * **`max_freq`**: The maximum proportion of documents a term can appear in. Setting `max_freq=0.8` (0.8 or 80%) would remove terms that appear in almost every document, acting almost like custom stop words for your specific dataset.
     * *Why use them?* To focus on terms that are neither too rare (potentially noise) nor too common (not distinctive).
 
 * **`ngrams`**: This allows you to treat **phrases** (sequences of words) as single "terms" in your DTM, rather than just individual words.
     * You provide a tuple like `(1, 2)`: `1` includes single words (unigrams), and `2` includes two-word phrases (bigrams). `(2, 3)` would include two- and three-word phrases.
     * *Why use it?* Captures collocations and common phrases that might carry more meaning than individual words (e.g., "dark forest" vs. "dark" and "forest" separately).
 
 #### Creating Our DTM
 
 Let's create our DTM. In this example, we'll start with the default settings for `stop_words` and `lemmatize` (which means stop words *will* be removed and lemmatization *will not* be applied by default in Lexos's DTM, but you can change it!). We'll use `nlp(doc)` to first process each raw text string into a `spaCy` document object.


In [7]:
# Create an instance of the DTM object
dtm = DTM()

# Process our documents with spaCy and then create the DTM.
# We'll use default settings for now, but you can uncomment and adjust parameters below!
dtm(
    docs=[nlp(doc) for doc in docs], # First, process each text with spaCy
    labels=labels,                     # Use our defined document labels
    # Here are some optional parameters you can uncomment and try:
    # stop_words=True,   # Remove common words like 'the', 'is', 'and' (True by default in Lexos DTM)
    # lemmatize=True,    # Convert words to their base form (e.g., 'running' -> 'run')
    # pos_filter=['NOUN', 'VERB', 'ADJ'], # Only include nouns, verbs, and adjectives
    # min_freq=2,        # Only include terms that appear at least 2 times overall
    # ngrams=(1, 2)      # Include single words (unigrams) and two-word phrases (bigrams)
)

print(f"DTM created with {dtm.to_df().shape[0]} documents and {dtm.to_df().shape[1]} unique terms.")


DTM created with 7713 documents and 4 unique terms.


### 5. Generating the Plotly Clustermap: Visualizing Relationships and Distances
 
 Now for the exciting part: generating and displaying your **Plotly Clustermap**! This powerful visualization combines a **heatmap** with **dendrograms** to help you understand the relationships between your documents.
 
 A clustermap visually reorganizes your data so that similar documents are placed closer together. It uses two dendrograms (one for rows, one for columns) to show the clustering, and a heatmap in the center to display the actual similarity or distance values.
 
 When we create the `PlotlyClustermap`, we need to tell it how to measure document similarity and how to organize the clusters. Here are the key parameters you can adjust:
 
 * **`dtm`**: This is our "linguistic spreadsheet" (`dtm`) that we created in the previous step. It's the essential input.
 
 * **`labels`**: This is the list of descriptive names for your documents (e.g., "Poe", "Lippard") that we defined earlier. These will appear along the axes of your clustermap.
 
 * **`metric`**: This tells the clustermap how to measure the "distance" or dissimilarity between your documents. Shorter distances mean more similar documents.
     * **`"euclidean"` (default)**: Think of this as the "straight-line" distance between two points on a graph. It's good for general comparisons but can be sensitive to the overall length of documents (longer documents might naturally have larger term counts, increasing their "distance").
     * **`"cosine"`**: Imagine each document as an arrow pointing in a specific linguistic "direction." Cosine similarity measures how much these arrows point in the same direction. If they point almost identically, the documents are very similar, even if one document is much longer than another. This is often an excellent choice for text analysis as it focuses on stylistic or thematic *direction* rather than raw word counts.
     * **`"cityblock"`** (also called Manhattan distance): Imagine moving on a city grid where you can only go along streets (no diagonal shortcuts). This distance is the sum of the absolute differences for each term between two documents. Useful when the individual differences in term counts are important.
     * Many other metrics are available (e.g., "jaccard", "chebyshev"). You can find a full list in the SciPy documentation for `scipy.spatial.distance.pdist`.
 
 * **`method`**: Once we've measured distances, this method determines how individual documents (or existing clusters of documents) are joined together to form larger branches and clusters in the dendrograms.
     * **`"average"` (default)**: When combining two clusters, this method considers the average distance between *all* pairs of documents in the two clusters. It tends to produce well-balanced clusters.
     * **`"single"`**: Joins clusters based on the *closest* pair of documents between them. This can sometimes lead to "chaining," where documents connect one after another, forming long, straggly branches.
     * **`"complete"`**: Joins clusters based on the *farthest* pair of documents between them. This tends to produce more compact, spherical clusters, ensuring all documents within a cluster are relatively similar to each other.
     * **`"ward"`**: This method aims to minimize the increase in "variance" (or spread) within clusters when they are merged. It tries to make clusters that are as "tight" and internally similar as possible. Often produces intuitive and well-structured clusters.
     * Many other methods are available. You can find a full list in the SciPy documentation for `scipy.cluster.hierarchy.linkage`.
 
 * **`colorscale`**: Determines the color scheme for the heatmap. You can choose from various built-in Plotly colorscales.
     * Examples: `"Viridis"` (default), `"Blues"`, `"Greys"`, `"Hot"`, `"RdBu"`, `"Jet"`.
     * The color intensity on the heatmap will represent the distance between documents: typically, darker/different colors show greater distance (less similarity), while lighter/similar colors show shorter distance (more similarity).
 
 * **`hide_upper`**: If `True`, the dendrogram at the top of the heatmap will be hidden.
 
 * **`hide_side`**: If `True`, the dendrogram on the left side of the heatmap will be hidden.
 
 * **`width` / `height`**: Control the overall dimensions of the plot in pixels.
 
 * **`title`**: Adds a title to your clustermap plot.
 
 * **`showfig`**: If `True`, the plot will be displayed immediately when you call the `PlotlyClustermap` instance. If `False` (default), you'll need to call `clustermap.show()` explicitly.
 
 Let's generate our first Plotly Clustermap!

In [8]:
# Create an instance of the PlotlyClustermap object
clustermap = PlotlyClustermap()

# Generate the Plotly Clustermap.
# Experiment with the parameters below!
try:
    clustermap(
        dtm=dtm,
        labels=labels,
        metric="euclidean",  # Try "cosine" for stylistic comparisons, or "cityblock"
        method="average",    # Try "ward" for compact clusters, or "complete"
        colorscale="Viridis",# Choose a color scheme for the heatmap (e.g., "Blues", "RdBu")
        width=800,           # Set width of the plot in pixels
        height=700,          # Set height of the plot in pixels
        title="Document Similarity Clustermap",
        showfig=True         # Display the figure in the notebook
    )

    # If showfig=False was used, you can explicitly show the figure like this:
    # clustermap.show()

except LexosException as e:
    print(f"An error occurred while generating the clustermap: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### 6. Interpreting Your Plotly Clustermap
 
 Now that you've generated the clustermap, let's talk about how to read it! A clustermap combines the best of two worlds: hierarchical clustering (dendrograms) and pairwise similarity representation (heatmap).
 
 * **Dendrograms (Top and Left)**:
     * The dendrogram on the **top** shows the hierarchical clustering of your documents based on their content (the rows of your DTM).
     * The dendrogram on the **left** shows the same clustering but rotated.
     * **Interpretation:** Just like with standalone dendrograms, shorter branches mean documents (or clusters) are more similar. The order of documents along the heatmap axes is determined by these dendrograms, grouping similar documents together.
 
 * **Heatmap (Center Square)**:
     * This square grid visually represents the **pairwise distances** between your documents. Each cell at the intersection of a row and a column represents the distance between two documents.
     * The **color** of each cell indicates the distance. The `colorscale` you chose determines what colors mean. For instance, with "Viridis," a yellow cell would represent a shorter distance (more similar), while a dark purple cell would represent a longer distance (less similar). With "RdBu," red might be high distance and blue low, or vice versa depending on the exact mapping.
     * The diagonal of the heatmap will always be the same color, usually representing zero distance, as a document has zero distance from itself.
     * **Interpretation:** Look for blocks of similarly colored cells. For example, if you see a bright yellow square where "Poe" and "Lippard" intersect, it means they are very similar. If a dark purple square appears between "Poe" and "Henry," it means they are quite dissimilar.
 
 **How to Read the Clustermap:**
 
 1.  **Find Clusters:** Start by examining the dendrograms. Identify branches that merge quickly, indicating closely related documents.
 2.  **Verify with Heatmap:** Once you've identified a cluster in the dendrogram (e.g., "Poe" and "Lippard" grouped together), look at the corresponding square on the heatmap where their rows and columns intersect. You should see colors indicating high similarity (low distance) within that cluster.
 3.  **Identify Outliers:** Documents that join the main clusters with very long branches in the dendrogram, or show consistently dark/dissimilar colors in the heatmap when compared to most other documents, are likely outliers.
 
 **What Your Clustermap Might Tell You:**
 
 * **Stylistic Groupings:** Does the heatmap show a strong block of similarity among authors from the same literary period or movement?
 * **Thematic Cohesion:** If your DTM focused on specific themes, do documents discussing similar themes cluster together?
 * **Influence or Divergence:** You might see how a text aligns with or diverges from others, giving insights into authorship, genre, or evolution of style.
 
 This interactive plot allows you to hover over cells to see exact distance values, which can be very helpful!


### 7. Customizing Your Visualization: Making the Clustermap Your Own
 
 The `lexos` `PlotlyClustermap` generates an interactive Plotly figure, giving you a lot of flexibility for customization beyond the initial parameters. You can adjust dimensions, hide dendrograms, and even control the colorscale.
 
 #### Adjusting Dimensions and Hiding Dendrograms
 
 You can change the `width` and `height` of the plot to make it larger or smaller. You can also hide the top or side dendrograms if you only want to focus on the heatmap or a single clustering view.


In [9]:
try:
    clustermap_custom = PlotlyClustermap()
    clustermap_custom(
        dtm=dtm,
        labels=labels,
        metric="cosine", # Using cosine for a different perspective
        method="ward",   # Using ward for compact clusters
        width=700,
        height=600,
        hide_upper=False, # Set to True to hide the top dendrogram
        hide_side=False,  # Set to True to hide the side dendrogram
        colorscale="Blues", # Try a different colorscale
        title="Clustermap: Cosine Distance, Ward Linkage (Blue Scale)",
        showfig=True
    )
except LexosException as e:
    print(f"An error occurred: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


### 8. Saving Your Clustermap: Keeping Your Results
 
 Once you're happy with your Plotly Clustermap, you'll likely want to save it as an interactive HTML file or a static image for reports, presentations, or simply for your records.
 
 Plotly figures are highly interactive when saved as HTML, allowing you to zoom, pan, and hover over data points in your saved file.
 
 * **`clustermap.write_html("filename.html")`**: Saves an interactive HTML file. This is generally recommended for Plotly figures.


In [ ]:
# Save as an interactive HTML file
try:
    # clustermap_custom.fig is the underlying Plotly figure object
    clustermap_custom.fig.write_html("my_clustermap_analysis.html")
    print("Clustermap saved as 'my_clustermap_analysis.html' (interactive).")
except LexosException as e:
    print(f"Could not save HTML: {e}")
except Exception as e:
    print(f"An unexpected error occurred while saving HTML: {e}")

Clustermap saved as 'my_clustermap_analysis.html' (interactive).


### 9. Troubleshooting

 **"My clustermap looks blank or I get errors about the DTM."**
 * **Verify DTM:** Ensure your `dtm` was created successfully in Section 4 and contains terms. If your documents are very short or very similar, the DTM might be sparse or have issues that prevent clustering.
 * **Check `showfig=True`**: Make sure you set `showfig=True` when calling the `PlotlyClustermap` instance, or explicitly call `clustermap.show()` after creating it.
 
 **"My documents don't cluster the way I expected!"**
 * **Experiment with `metric` and `method`**: Different distance metrics and linkage methods will highlight different types of similarity. `"cosine"` is often excellent for stylistic comparisons in text. `"ward"` or `"average"` are common and effective linkage methods.
 * **Adjust DTM pre-processing**: The content of your DTM directly influences the clustering. Try different combinations of `stop_words`, `lemmatize`, `pos_filter`, `min_freq`, and `ngrams` when creating your DTM.
 
 **"I'm getting errors about Lexos modules not found."**
 * Ensure you have installed `lexos`. If not, run `pip install lexos` in your command line.
 * If you are running this notebook from a cloned `lexos` repository, make sure your Python environment is set up correctly to find the `lexos` package (e.g., by running `pip install -e .` from the `lexos` root directory, or by adding the `lexos` directory to your Python path).
 
 **"I can't save static images (e.g., PNG)."**
 * Plotly requires an additional library called `kaleido` to export static images. Install it by running `pip install kaleido` in your command line.